## Training scheme for https://link.springer.com/content/pdf/10.1007/978-3-031-93688-3_19.pdf
## But using a GCN

- Building training data for detector

In [ ]:
# Frame-level punch / no-punch labels from Excel (V1–V10), aligned to source RGB MP4s.
# 0 = no punch, 1 = inside some annotated punch interval (inclusive 1-based [start,end]).
# Gaps between clips (e.g. frames 5–6 between [1–4] and [7–9]) stay 0 — full timeline covered.
# Also saves sliding-window rows for GCNDetector-style training (window_length=T, label=1 if any punch frame in window).

from pathlib import Path

import cv2
import numpy as np

from preprocess import _find_video, _load_annotations

_REPO = Path.cwd().resolve()
assert (_REPO / "preprocess.py").exists(), "Run this notebook with working directory = repo root (pose/)."

_ANNOT = _REPO / "Dataset" / "Annotation_files"
_OUT = _REPO / "Dataset" / "detection_frame_labels"
_OUT.mkdir(parents=True, exist_ok=True)

WINDOW_LENGTH = 11  # temporal length for GCNDetector / Baghel-style windows
STRIDE = 1
# Use only the first N frames of each MP4 in this npz (None = full video).
MAX_VIDEO_FRAMES = 256


def _video_frame_count(path: Path) -> int:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return 0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return max(0, n)


def _frame_punch_binary(num_frames: int, annotations: list[tuple[int, int, str]]) -> np.ndarray:
    """One label per video frame. Annotations use the same 1-based inclusive convention as preprocess.extract_landmarks."""
    y = np.zeros(num_frames, dtype=np.uint8)
    for s, e, _ in annotations:
        if e < s:
            continue
        s0 = max(0, s - 1)
        e_excl = min(num_frames, e)  # exclusive end index: 1-based inclusive e → indices … e-1
        if s0 < e_excl:
            y[s0:e_excl] = 1
    return y


def _sliding_windows(frame_y: np.ndarray, window: int, stride: int) -> tuple[np.ndarray, np.ndarray]:
    """window_is_punch[k] = 1 iff max(frame_y[t:t+window]) == 1 (any punch frame in the window)."""
    if len(frame_y) < window:
        return np.zeros(0, dtype=np.int64), np.zeros(0, dtype=np.uint8)
    starts = np.arange(0, len(frame_y) - window + 1, stride, dtype=np.int64)
    win_y = np.array(
        [int(frame_y[t : t + window].max() > 0) for t in starts],
        dtype=np.uint8,
    )
    return starts, win_y


for i in range(1, 11):
    ver = f"V{i}"
    vid = _find_video(ver)
    xlsx = _ANNOT / f"{ver}.xlsx"
    if vid is None or not xlsx.exists():
        print(f"[skip] {ver}: missing video or {xlsx.name}")
        continue

    ann = _load_annotations(xlsx)
    F_full = _video_frame_count(vid)
    if F_full == 0:
        print(f"[skip] {ver}: zero frames")
        continue

    frame_y = _frame_punch_binary(F_full, ann)
    if MAX_VIDEO_FRAMES is not None:
        F = min(F_full, int(MAX_VIDEO_FRAMES))
        frame_y = frame_y[:F]
    else:
        F = F_full

    w_starts, w_y = _sliding_windows(frame_y, WINDOW_LENGTH, STRIDE)

    cap = cv2.VideoCapture(str(vid))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    cap.release()

    intervals = (
        np.array([[s, e] for s, e, _ in ann], dtype=np.int32)
        if ann
        else np.zeros((0, 2), dtype=np.int32)
    )

    out = _OUT / f"{ver}_detection.npz"
    np.savez_compressed(
        out,
        punch_frame_binary=frame_y,
        num_frames=np.int32(F),
        fps=np.float32(fps),
        window_length=np.int32(WINDOW_LENGTH),
        stride=np.int32(STRIDE),
        window_starts=w_starts,
        window_is_punch=w_y,
        punch_intervals_1based=intervals,
        version=np.array(ver),
        source_video=np.array(str(vid)),
    )

    print(
        f"{ver}: frames={F}  punch_frames={int(frame_y.sum())}  "
        f"windows={len(w_y)}  punch_windows={int(w_y.sum())}  -> {out.relative_to(_REPO)}"
    )

print(f"\nDone. Labels under {_OUT.relative_to(_REPO)}/")
print("Pair each window start index with pose windows [N,M,T,V,C] when you stack skeleton clips.")

### Outputs (`Dataset/detection_frame_labels/`)

For each `V1` … `V10`, **`{ver}_detection.npz`** contains:

| Key | Meaning |
|-----|--------|
| `punch_frame_binary` | `(num_frames,)` uint8 — 0/1 for every frame of the MP4 |
| `num_frames`, `fps` | Video length and FPS |
| `punch_intervals_1based` | `(N, 2)` Excel intervals `[start, end]` (inclusive, 1-based) |
| `window_length`, `stride` | Defaults 11 and 1 |
| `window_starts` | Start indices `t` for windows `[t : t+window_length)` |
| `window_is_punch` | 1 if **any** frame in that window is punch (for detector targets) |
| `version`, `source_video` | Strings |

**GCNDetector** expects clips `[N, M, T, V, C]` with `T = window_length`.

## Punch detector training (`GCNDetector`)

1. **Build window cache** (next cell): one MediaPipe pass over the RGB video, then stack 11-frame clips aligned with `window_starts` / `window_is_punch`. Use `max_windows` to cap size (balanced punch vs no-punch). *Slow* on long videos.
2. **Train** (following cell): `GCNDetector` (BoxingVI 12 joints), `BCELoss`, AdamW.

Restart the kernel after editing `GCN.py` or `detector_data.py`.

In [1]:
# --- Build pose cache for one workbook (repeat or loop V1–V10) ---
from pathlib import Path

from preprocess import _find_video
from detector_data import build_detector_windows_npz

_REPO = Path.cwd().resolve()
DET_VER = "V1"

DET_NPZ = _REPO / "Dataset" / "detection_frame_labels" / f"{DET_VER}_detection.npz"
OUT_CACHE = _REPO / "Dataset" / "detector_training" / f"{DET_VER}_windows.npz"
VIDEO = _find_video(DET_VER)

MAX_WINDOWS = 8000  # None = keep every window after pose extract (very large)

assert DET_NPZ.exists(), "Run the preprocessing cell first."
assert VIDEO is not None and VIDEO.exists(), f"No MP4 for {DET_VER}"

build_detector_windows_npz(
    DET_NPZ,
    VIDEO,
    OUT_CACHE,
    max_windows=MAX_WINDOWS,
    balance=True,
    seed=42,
)
print(f"Saved detector cache → {OUT_CACHE.relative_to(_REPO)}")

I0000 00:00:1778171825.047182   38029 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1778171825.086822   38047 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.95.05), renderer: NVIDIA GeForce RTX 5070/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778171825.112845   38033 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778171825.126073   38037 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778171825.151760   38038 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Frame:  0 / 46559
Frame:  100 / 46559
Frame:  200 / 46559
Frame:  300 / 46559
Frame:  400 / 46559
Frame:  500 / 46559
Frame:  600 / 46559
Frame:  700 / 46559
Frame:  800 / 46559
Frame:  900 / 46559
Frame:  1000 / 46559
Frame:  1100 / 46559
Frame:  1200 / 46559
Frame:  1300 / 46559
Frame:  1400 / 46559
Frame:  1500 / 46559
Frame:  1600 / 46559
Frame:  1700 / 46559
Frame:  1800 / 46559
Frame:  1900 / 46559
Frame:  2000 / 46559
Frame:  2100 / 46559
Frame:  2200 / 46559
Frame:  2300 / 46559
Frame:  2400 / 46559
Frame:  2500 / 46559
Frame:  2600 / 46559
Frame:  2700 / 46559
Frame:  2800 / 46559
Frame:  2900 / 46559
Frame:  3000 / 46559
Frame:  3100 / 46559
Frame:  3200 / 46559
Frame:  3300 / 46559
Frame:  3400 / 46559
Frame:  3500 / 46559
Frame:  3600 / 46559
Frame:  3700 / 46559
Frame:  3800 / 46559
Frame:  3900 / 46559
Frame:  4000 / 46559
Frame:  4100 / 46559
Frame:  4200 / 46559
Frame:  4300 / 46559
Frame:  4400 / 46559
Frame:  4500 / 46559
Frame:  4600 / 46559
Frame:  4700 / 46559
Fram

In [ ]:
# --- Train GCNDetector on cached windows ---
import torch
from pathlib import Path

from torch.utils.data import ConcatDataset, DataLoader, random_split

from GCN import (
    BOXINGVI_BONE_PAIRS,
    BOXINGVI_CENTER_JOINT,
    BOXINGVI_GRAPH_EDGES,
    GCNDetector,
    NUM_BOXINGVI_JOINTS,
)
from detector_data import DetectorWindowNpzDataset

_REPO = Path.cwd().resolve()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build one `{ver}_windows.npz` per entry (detector cache cell), then list them here.
DATA_VERS = ["V7"]
_CACHEDIR = _REPO / "Dataset" / "detector_training"
_CACHES = [_CACHEDIR / f"{v}_windows.npz" for v in DATA_VERS]
for p in _CACHES:
    assert p.exists(), f"Missing cache — build it first: {p}"

# Overfitting sanity check: strip regularization + train longer → train acc should approach 1.0.
# Set False when tuning for validation/generalization again.
OVERFIT_SANITY = True

if OVERFIT_SANITY:
    EPOCHS = 80
    LR = 3e-3
    WEIGHT_DECAY = 0.0
    BACKBONE_DROPOUT = 0.0
else:
    EPOCHS = 15
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    BACKBONE_DROPOUT = 0.1

BATCH_SIZE = 48
VAL_FRAC = 0.15
SEED = 42

torch.manual_seed(SEED)
_parts = [DetectorWindowNpzDataset(p) for p in _CACHES]
full_ds = ConcatDataset(_parts)
print(f"ConcatDataset: {len(DATA_VERS)} files, {len(full_ds)} windows total")
for v, ds in zip(DATA_VERS, _parts):
    print(f"  {v}: {len(ds)}")
if OVERFIT_SANITY:
    print(
        "OVERFIT_SANITY: weight_decay=0, backbone dropout=0 — expect train acc → ~1.0 "
        "(val may degrade)."
    )

n_val = max(1, int(round(VAL_FRAC * len(full_ds))))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = GCNDetector(
    in_channels=2,
    num_joints=NUM_BOXINGVI_JOINTS,
    bone_pairs=BOXINGVI_BONE_PAIRS,
    backbone_kwargs={
        "edges": BOXINGVI_GRAPH_EDGES,
        "center": BOXINGVI_CENTER_JOINT,
        "dropout": BACKBONE_DROPOUT,
        "data_bn": True,
    },
    dropout=0,
).to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = torch.nn.BCELoss()


@torch.no_grad()
def evaluate(loader: DataLoader) -> tuple[float, float]:
    model.eval()
    total, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        p = model(x)
        total += loss_fn(p, y).item() * y.size(0)
        pred = (p >= 0.5).float()
        correct += (pred == y).sum().item()
        n += y.size(0)
    return total / max(n, 1), correct / max(n, 1)


for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss = 0.0
    run_correct = 0
    run_n = 0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        p = model(x)
        loss = loss_fn(p, y)
        loss.backward()
        opt.step()
        run_loss += loss.item() * y.size(0)
        run_correct += ((p >= 0.5).float() == y).sum().item()
        run_n += y.size(0)

    tr_loss = run_loss / max(run_n, 1)
    tr_acc = run_correct / max(run_n, 1)
    va_loss, va_acc = evaluate(val_dl)
    print(
        f"epoch {epoch:02d}/{EPOCHS}  train loss {tr_loss:.4f} acc {tr_acc:.3f}  "
        f"val loss {va_loss:.4f} acc {va_acc:.3f}"
    )

tr_final_loss, tr_final_acc = evaluate(train_dl)
print(
    f"Train (eval mode, same split): loss {tr_final_loss:.4f} acc {tr_final_acc:.4f}"
)

_det_tag = "_".join(DATA_VERS)
ckpt = _REPO / "checkpoints" / f"gcn_detector_{_det_tag}.pt"
ckpt.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state": model.state_dict(),
        "data_vers": DATA_VERS,
        "caches": [str(p) for p in _CACHES],
    },
    ckpt,
)
print(f"Checkpoint → {ckpt.relative_to(_REPO)}")


ConcatDataset: 10 files, 80000 windows total
  V1: 8000
  V2: 8000
  V3: 8000
  V4: 8000
  V5: 8000
  V6: 8000
  V7: 8000
  V8: 8000
  V9: 8000
  V10: 8000
OVERFIT_SANITY: weight_decay=0, backbone dropout=0 — expect train acc → ~1.0 (val may degrade).
epoch 01/80  train loss 0.5954 acc 0.661  val loss 0.5791 acc 0.684
epoch 02/80  train loss 0.5329 acc 0.721  val loss 0.5245 acc 0.728
epoch 03/80  train loss 0.5027 acc 0.744  val loss 0.4750 acc 0.765
epoch 04/80  train loss 0.4832 acc 0.758  val loss 0.4681 acc 0.766
epoch 05/80  train loss 0.4684 acc 0.765  val loss 0.4724 acc 0.767
epoch 06/80  train loss 0.4601 acc 0.769  val loss 0.4421 acc 0.780
epoch 07/80  train loss 0.4499 acc 0.776  val loss 0.4348 acc 0.783
epoch 08/80  train loss 0.4417 acc 0.781  val loss 0.4353 acc 0.785
epoch 09/80  train loss 0.4337 acc 0.785  val loss 0.4421 acc 0.777
epoch 10/80  train loss 0.4287 acc 0.788  val loss 0.4133 acc 0.793
epoch 11/80  train loss 0.4229 acc 0.791  val loss 0.4209 acc 0.787


In [5]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable_params)

4725537


## Punch-type classification (`GCNClassifier`)

Trains **separately** from the punch/no-punch detector. Uses **`Dataset/landmarks.npz`**: each row is one annotated clip with a **six-class** punch label.

- **Windows**: `prepare_windows(..., window=CLF_WINDOW)` gives `(N, 20, 12, 2)` → batch shape **`[N, 1, 20, 12, 2]`**.
- **Labels**: strings in the npz (`Jab`, `Cross`, …) map to indices **`0 … 5`** in **`GCN.PUNCH_CLASSES`** order.

Generate **`landmarks.npz`** first (e.g. `python preprocess.py extract` from repo root).

In [1]:
# --- Train GCNClassifier (6 punch types) from landmarks.npz ---
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

from GCN import (
    BOXINGVI_BONE_PAIRS,
    BOXINGVI_CENTER_JOINT,
    BOXINGVI_GRAPH_EDGES,
    GCNClassifier,
    NUM_BOXINGVI_JOINTS,
    PUNCH_CLASSES,
)
from preprocess import prepare_windows

_REPO = Path.cwd().resolve()
NPZ_PATH = _REPO / "Dataset" / "landmarks.npz"

_LABEL_TO_IDX = {
    "Cross": PUNCH_CLASSES.index("cross"),
    "Jab": PUNCH_CLASSES.index("jab"),
    "Lead Hook": PUNCH_CLASSES.index("lead_hook"),
    "Rear Hook": PUNCH_CLASSES.index("rear_hook"),
    "Lead Uppercut": PUNCH_CLASSES.index("lead_uppercut"),
    "Rear Uppercut": PUNCH_CLASSES.index("rear_uppercut"),
}

CLF_WINDOW = 20
EPOCHS = 40
BATCH_SIZE = 64
LR = 1e-3
VAL_FRAC = 0.2
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert NPZ_PATH.exists(), "Missing landmarks.npz — run preprocess extraction first."

data = np.load(NPZ_PATH, allow_pickle=True)
seqs = data["sequences"]
labels = data["labels"]

X_win, y_str = prepare_windows(seqs, labels, window=CLF_WINDOW)
X = np.nan_to_num(X_win.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

keep_rows: list[int] = []
y_list: list[int] = []
for i, s in enumerate(y_str):
    s = str(s).strip()
    if s not in _LABEL_TO_IDX:
        continue
    keep_rows.append(i)
    y_list.append(_LABEL_TO_IDX[s])
X = X[keep_rows]
y_idx = np.array(y_list, dtype=np.int64)
if len(keep_rows) < len(y_str):
    print(f"Kept {len(keep_rows)}/{len(y_str)} clips (dropped unknown labels)")


class ClfDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray):
        self.x = x
        self.y = y.astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        t = torch.from_numpy(self.x[i]).unsqueeze(0).unsqueeze(0)
        return t, torch.tensor(self.y[i], dtype=torch.long)


idx_train, idx_val = train_test_split(
    np.arange(len(y_idx)),
    test_size=VAL_FRAC,
    random_state=SEED,
    stratify=y_idx,
)
train_ds = ClfDataset(X[idx_train], y_idx[idx_train])
val_ds = ClfDataset(X[idx_val], y_idx[idx_val])

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = GCNClassifier(
    num_classes=len(PUNCH_CLASSES),
    in_channels=2,
    num_joints=NUM_BOXINGVI_JOINTS,
    bone_pairs=BOXINGVI_BONE_PAIRS,
    backbone_kwargs={
        "edges": BOXINGVI_GRAPH_EDGES,
        "center": BOXINGVI_CENTER_JOINT,
        "dropout": 0.1,
        "data_bn": True,
    },
    dropout=0.3,
).to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
crit = torch.nn.CrossEntropyLoss()


@torch.no_grad()
def eval_epoch(loader):
    model.eval()
    tot, correct, n = 0.0, 0, 0
    all_p, all_t = [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = crit(logits, yb)
        tot += loss.item() * yb.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        n += yb.size(0)
        all_p.append(pred.cpu())
        all_t.append(yb.cpu())
    all_p = torch.cat(all_p).numpy()
    all_t = torch.cat(all_t).numpy()
    return tot / max(n, 1), correct / max(n, 1), all_p, all_t


best_acc = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss, run_ok, run_n = 0.0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        run_loss += loss.item() * yb.size(0)
        run_ok += (logits.argmax(1) == yb).sum().item()
        run_n += yb.size(0)
    sched.step()

    va_loss, va_acc, _, _ = eval_epoch(val_dl)
    if va_acc > best_acc:
        best_acc = va_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"epoch {epoch:02d}/{EPOCHS}  train loss {run_loss/max(run_n,1):.4f} acc {run_ok/max(run_n,1):.3f}  "
        f"val loss {va_loss:.4f} acc {va_acc:.3f}"
    )

if best_state is not None:
    model.load_state_dict(best_state)
_, _, vp, vt = eval_epoch(val_dl)
print("\nClassification report (val, best checkpoint):")
print(
    classification_report(
        vt,
        vp,
        target_names=[p.replace("_", " ").title() for p in PUNCH_CLASSES],
        zero_division=0,
    )
)
print("Confusion matrix:\n", confusion_matrix(vt, vp))

ckpt = _REPO / "checkpoints" / "gcn_classifier.pt"
ckpt.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state": best_state,
        "punch_classes": PUNCH_CLASSES,
        "label_map": _LABEL_TO_IDX,
        "window": CLF_WINDOW,
    },
    ckpt,
)
print(f"Checkpoint → {ckpt.relative_to(_REPO)}")

epoch 01/40  train loss 1.7268 acc 0.243  val loss 1.7146 acc 0.251
epoch 02/40  train loss 1.7105 acc 0.247  val loss 1.6941 acc 0.251
epoch 03/40  train loss 1.6905 acc 0.259  val loss 1.6597 acc 0.280


KeyboardInterrupt: 